# Track A: evaluate the saved evidence, step by step

This notebook never reruns clustering or quantum jobs. The gap definitions,
raw-shot weighting, feasibility filtering, best-partition selection, routing
cache and CSV assembly are executable code below.

Only input validation, common encoding/decoding (derived in the quantum
notebook), exact fixed-group routing (defined in the classical notebook),
file utilities, and PDF page layout remain shared.

## Before the tables: how should we judge an answer?

An answer is **feasible** if it serves every customer exactly once and
overloads no vehicle. A low reported cost is not useful if these rules fail.

A **reference** is the best cost found by our exact classical search, which
checks every relevant possibility. Here it is a proven optimum, not a guess.
There are separate references for customer-group compactness and delivery
distance because these are different goals.

A **gap** says how much worse an answer is than its reference. If the reference
cost is 100 and an answer costs 105, its gap is 5%. Lower is better; zero
matches the reference.

**Feasibility rate** and **hit probability** answer different questions:
if 5 of 100 shots are legal and 2 also match the reference closely enough,
the feasibility rate is 5% and the hit probability is 2%, **not 40%**.
Discarding bad shots must not make the method look more successful.

An empty gap means "no usable answer" (or an undefined percentage), not zero
error. Timings and numerical limits describe a local experiment, not evidence
that a quantum computer is faster.

In [ ]:
from pathlib import Path
import json
import sys
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "track_a" / "inputs" / "seeds_s.json").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from track_a.common.instances import OUTPUT_PATH, Instance, ensure_instances, feasible, validate
from track_a.common.analysis import fingerprint, save
from track_a.common.models import ENCODING

instances = ensure_instances()
print("Processing only the first demand vector:", instances[0].demands)
output = OUTPUT_PATH
shots = 256
seed = 0

import csv
from track_a.common.classical import Routing
from track_a.common.models import decode, objective
from track_a.common.report import write_report

GAP_TOLERANCE_PERCENT = 0.1

## 1. Define percentage gaps, including zero references

Clustering and CVRP use their own exact references. Negative gaps
are not clipped. Equal zero costs have zero gap; a nonzero cost against a
zero reference has undefined percentage gap.

**Reading the calculation:** subtract the
reference cost from the answer cost, divide by the reference, then multiply
by 100. A reference of zero cannot be used as a divisor, so that case needs
an explicit rule rather than an accidental infinity.

In [ ]:
def gap(cost: float, reference: float) -> float | None:
    if reference == 0:
        return 0.0 if cost == 0 else None
    return 100 * (cost - reference) / reference

## 2. Evaluate raw shots without repair

Only original-constraint feasible shots contribute hits, but
**all raw shots** form the denominator at 0.1%, 1% and 5% thresholds.
Route each distinct feasible partition exactly, preserving the original shot
counts. Best clustering and best CVRP gaps are minimized separately and may
come from different partitions. No feasible shots means undefined gaps.

**Why keep raw counts?** If one good bitstring appears
20 times, it contributes 20 successful shots even though we only need to
calculate its route once. A 0.1% hit threshold means "within 0.1 percent of
the reference," not "the method is right with probability 0.1 percent."
We do not edit an illegal string into a legal one before counting success.

In [ ]:
def metrics(instance: Instance, counts: dict[str, int], reference: dict,
            routing: Routing) -> tuple[dict, list[dict]]:
    if any(type(count) is not int or count <= 0 for count in counts.values()):
        raise ValueError("Raw counts must be positive integers")
    total = sum(counts.values())
    feasible_count = 0
    evaluations: dict[tuple[int, ...], dict] = {}
    hits = {f"{name}_hit_probability_{threshold:g}pct": 0
            for name in ("model", "cvrp") for threshold in (0.1, 1.0, 5.0)}
    routing_seconds = 0.0
    for bitstring, count in counts.items():
        labels = decode(instance, bitstring)
        if labels is None:
            continue
        feasible_count += count
        if labels not in evaluations:
            start = perf_counter()
            routed = routing.evaluate(instance, labels)
            routing_seconds += perf_counter() - start
            value = objective(instance, labels)
            evaluations[labels] = {
                "labels": list(labels), "model_cost": value, "route_cost": routed["cost"],
                "model_gap": gap(value, reference["clustering"]["cost"]),
                "cvrp_gap": gap(routed["cost"], reference["cvrp"]["cost"]),
                "tours": routed["tours"], "shots": 0,
            }
        item = evaluations[labels]
        item["shots"] += count
        for name in ("model", "cvrp"):
            for threshold in (0.1, 1.0, 5.0):
                if item[f"{name}_gap"] is not None and item[f"{name}_gap"] <= threshold:
                    hits[f"{name}_hit_probability_{threshold:g}pct"] += count
    best_model = min(evaluations.values(), key=lambda e: e["model_cost"], default=None)
    best_route = min(evaluations.values(), key=lambda e: e["route_cost"], default=None)
    return {
        "actual_shots": total, "feasible_shots": feasible_count,
        "feasibility_rate": feasible_count / total if total else None,
        "best_model_gap": best_model["model_gap"] if best_model else None,
        "best_cvrp_gap": best_route["cvrp_gap"] if best_route else None,
        "best_model_labels": best_model["labels"] if best_model else None,
        "best_cvrp_labels": best_route["labels"] if best_route else None,
        "routing_seconds": routing_seconds,
        **{key: value / total if total else None for key, value in hits.items()},
    }, list(evaluations.values())

## 3. Reuse exact group tours across methods

Routing is evaluation only: fixed groups, exact depot tours, no reassignment
or optimizer feedback. The cache key includes geometry and depot.

In [ ]:
_ROUTING: dict[tuple, Routing] = {}

def _routing_for(instance: Instance) -> Routing:
    key = instance.coordinates, instance.depot
    if key not in _ROUTING:
        _ROUTING[key] = Routing(instance)
    return _ROUTING[key]

## 4. Join evidence, validate hashes, cache evaluations, and export CSV

Every run retains its instance, encoding and configuration IDs.
Reject incompatible inputs or wrong shot budgets. Reuse per-run routing
evaluations only when the run hash matches. Failed emulations retain their
status and never acquire success metrics. This loop invokes no optimizer.

**Why compare hashes?** A hash is a compact
fingerprint of saved data. We use it to catch accidentally mixing results
from different input coordinates or demands. This is bookkeeping for a fair
comparison, not another optimization algorithm. CSV is the ordinary table
format you can open in a spreadsheet.

In [ ]:
def summarize(output: Path) -> list[dict]:
    """Read saved evidence only: never invoke an optimizer or emulator."""
    rows = []
    for path in sorted((output / "runs").glob("*.json")):
        run = json.loads(path.read_text())
        instance = Instance(
            **{**run["instance"], "coordinates": tuple(map(tuple, run["instance"]["coordinates"])),
               "demands": tuple(run["instance"]["demands"]),
               "certificate": tuple(run["instance"]["certificate"]),
               "depot": tuple(run["instance"]["depot"])}
        )
        validate(instance)
        ref = json.loads((output / "references" / f"{instance.id}.json").read_text())
        row = {key: run.get(key) for key in (
            "instance_id", "method", "seed", "encoding", "configuration_id", "instance_hash",
            "status", "shot_budget", "atom_count", "clustering_seconds",
            "embedding_seconds", "emulation_seconds", "pair_relative_error",
            "linear_relative_error", "scale", "error",
        )}
        row.update(instance.ratios)
        row.update({"certificate_valid": True, "model_reference": ref["clustering"]["cost"],
                    "cvrp_reference": ref["cvrp"]["cost"],
                    "reference_status": ref["status"], "reference_seconds": ref["seconds"],
                    "model_reference_status": ref["clustering"]["status"],
                    "cvrp_reference_status": ref["cvrp"]["status"],
                    "gap_tolerance_percent": GAP_TOLERANCE_PERCENT})
        if run["encoding"] != ENCODING or run["instance_hash"] != fingerprint(run["instance"]):
            raise ValueError(f"Incompatible encoding or modified instance in {path}")
        if ref["instance_hash"] != run["instance_hash"]:
            raise ValueError(f"Reference-instance mismatch in {path}")
        cache_path = output / "evaluations" / path.name
        if run["status"] == "completed":
            if sum(run["counts"].values()) != run["shot_budget"]:
                raise ValueError(f"Shot budget mismatch in {path}")
            cached = json.loads(cache_path.read_text()) if cache_path.exists() else None
            if cached is not None and cached["run_hash"] == fingerprint(run):
                measured = cached["metrics"]
            else:
                measured, evaluations = metrics(instance, run["counts"], ref, _routing_for(instance))
                save(cache_path, {"run_hash": fingerprint(run), "metrics": measured,
                                  "partitions": evaluations})
            row.update(measured)
        rows.append(row)
    if not rows:
        raise ValueError(f"No saved runs found in {output / 'runs'}")
    fields = sorted(set().union(*(row.keys() for row in rows)))
    with (output / "results.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    save(output / "summary.json", rows)
    return rows

In [ ]:
rows = summarize(output)
results = pd.DataFrame(rows)
display(results[["instance_id", "method", "seed", "status", "shot_budget",
                 "feasibility_rate", "best_model_gap", "best_cvrp_gap",
                 "model_reference_status", "cvrp_reference_status"]])

## 5. Plot the actual atom layout and pulse from a saved run

Use the chosen run's pulse, not a newly fitted layout or a new
optimization. The saved matrices include unwanted interactions as well as
target couplings.

**What each plot means:** the layout labels show
which bit each atom represents; they do not show customer locations.
The other plots show the laser drive and detuning versus time. Saved
embedding errors indicate how closely this layout matches the intended
cost, independently of how good its measured answers were.

In [ ]:
def plot_evidence(output: Path, instance_id: str = "s-0", seed: int = 0):
    import matplotlib.pyplot as plt

    layout = json.loads((output / "embeddings" / f"{instance_id}-{seed}.json").read_text())
    run = json.loads((output / "runs" / f"{instance_id}-quantum-global-{seed}.json").read_text())
    pulse = run["pulse"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    coordinates = layout["coordinates_um"]
    axes[0].scatter(*zip(*coordinates))
    for name, (x, y) in zip(layout["variable_order"], coordinates, strict=True):
        axes[0].annotate(name, (x, y), fontsize=7)
    axes[0].set(xlabel="x (um)", ylabel="y (um)", title="Legal atom layout", aspect="equal")
    from pulser.waveforms import InterpolatedWaveform, RampWaveform
    amplitude = InterpolatedWaveform(pulse["duration_ns"], [0, pulse["amplitude"], 0])
    detuning = RampWaveform(pulse["duration_ns"], pulse["initial_detuning"], layout["final_detuning"])
    axes[1].plot(amplitude.samples.as_array(detach=True))
    axes[1].set(xlabel="time (ns)", ylabel="rad/us", title="Rabi amplitude")
    axes[2].plot(detuning.samples.as_array(detach=True))
    axes[2].set(xlabel="time (ns)", ylabel="rad/us", title="Global detuning")
    fig.tight_layout()
    return fig

In [ ]:
display(pd.DataFrame(json.loads((output / "instances.json").read_text()))[
    ["tier", "instance_seed", "demands", "rho", "largest_demand_ratio", "certificate_valid"]])
display(results[["instance_id", "method", "atom_count", "pair_relative_error",
                 "linear_relative_error", "model_hit_probability_0.1pct",
                 "cvrp_hit_probability_0.1pct"]])
plot_evidence(output, instances[0].id, seed)

## 6. Compare methods, inspect limits, and generate the four-page report

The constrained baseline repeats a deterministic solution; its rate is not an
independent-shot success estimate. QUBO-SA, random bits and quantum use the
same decoding/filtering and sample budget. Timings distinguish reference,
clustering, embedding, emulation and exact routing work.

In [ ]:
display(results.groupby(["method", "status"], dropna=False).agg(
    instances=("instance_id", "count"), mean_feasibility=("feasibility_rate", "mean"),
    mean_model_gap=("best_model_gap", "mean"), mean_cvrp_gap=("best_cvrp_gap", "mean")))
display(results[["instance_id", "method", "clustering_seconds", "embedding_seconds",
                 "emulation_seconds", "routing_seconds", "reference_seconds"]])
print("Checkpoint:", json.loads((output / "checkpoint.json").read_text()))
write_report(output, rows)
print("Saved:", output / "results.csv", output / "report.pdf")

## What did not work, limitations and AI disclosure

A dense binary-slack QUBO is not generally exactly embeddable with positive
2D interactions and a shared detuning. Inspect both saved errors; do not
interpret target correctness as physical correctness. Low feasible-shot rates
and failures remain in the tables, without fabricated gaps or hidden repairs.

This baseline tests one geometry and the supplied demand vectors, one default
sampling seed and two pulse candidates. Calibration uses S-0, not held-out
data. Noise robustness, time-step convergence, DMM and M/L scaling are deferred.
There is no quantum-speedup claim.

**AI use:** an AI assistant using Copilot SDK in VS Code assisted with the
implementation, notebook explanations and report. Numerical results are
actual saved local outputs and still require human review. See [MODEL.md](docs/MODEL.md)
for the full derivations and the separate report for the four-page summary.